In [1]:
import csv
import random
from datetime import datetime, timedelta

# 1. Master Menu Asli Nasi Kampus Samarinda (ID 1 - 14 sesuai urutan INSERT)
menus = [
    {"id": 1, "nama": "Paket 1", "kategori": "Nasi Kotak", "harga": 12000},
    {"id": 2, "nama": "Paket 2", "kategori": "Nasi Kotak", "harga": 15000},
    {"id": 3, "nama": "Paket 2 Ayam Bumbu", "kategori": "Nasi Kotak", "harga": 17000},
    {"id": 4, "nama": "Paket 3", "kategori": "Nasi Kotak", "harga": 17000},
    {"id": 5, "nama": "Paket 4", "kategori": "Nasi Kotak", "harga": 20000},
    {"id": 6, "nama": "Paket 5", "kategori": "Nasi Kotak", "harga": 25000},
    {"id": 7, "nama": "Paket Spesial", "kategori": "Nasi Kotak", "harga": 30000},
    {"id": 8, "nama": "Paket Bebek", "kategori": "Nasi Kotak", "harga": 33000},
    {"id": 9, "nama": "Snackbox 1", "kategori": "Snack Box", "harga": 5000},
    {"id": 10, "nama": "Snack Box 2", "kategori": "Snack Box", "harga": 7000},
    {"id": 11, "nama": "Snack Box 3", "kategori": "Snack Box", "harga": 10000},
    {"id": 12, "nama": "Snack Box 4", "kategori": "Snack Box", "harga": 12000},
    {"id": 13, "nama": "Snackbox 5", "kategori": "Snack Box", "harga": 15000},
    {"id": 14, "nama": "Snack Box Premium", "kategori": "Snack Box", "harga": 25000},
]

# 2. Generate 120 Pelanggan Unik (Instansi, Organisasi Kampus Unmul, Umum Samarinda)
prefix_nama = [
    "Himpunan Mahasiswa", "BEM", "UKM", "Panitia Seminar", "Ibu", "Bapak",
    "Kantor Dinas", "Bank Kaltimtara", "RSUD AWS", "Fakultas Teknik", "FEB Unmul",
    "FKIP Unmul", "Komunitas", "Pengajian", "Karang Taruna"
]
suffix_nama = [
    "Sistem Informasi", "Informatika", "Hukum", "Ekonomi", "Rina", "Hendra",
    "Sari", "Yusuf", "Dewi", "Rizky", "Nadia", "Fajar", "Samarinda Seberang",
    "Loa Bakung", "Sempaja", "Air Hitam", "Vorvo", "Antasari"
]

pelanggan_list = []
for i in range(1, 121):
    nama = f"{random.choice(prefix_nama)} {random.choice(suffix_nama)} {i}"
    wa = f"08{random.choice(['12', '13', '52', '53', '21', '22'])}{random.randint(10000000, 99999999)}"
    alamat = random.choice([
        "Jl. Sambaliung, Kampus Gunung Kelua, Samarinda",
        "Jl. P. Antasari, Samarinda Ulu",
        "Jl. Ahmad Yani, Sungai Pinang, Samarinda",
        "Jl. Juanda, Air Hitam, Samarinda",
        "Jl. PM Noor, Sempaja Selatan, Samarinda",
        "Jl. Slamet Riyadi, Karang Asam, Samarinda"
    ])
    pelanggan_list.append((i, nama, wa, alamat))

# 3. Generate 500 Pesanan & Detail Pesanan (Rentang Sept - Nov 2026)
start_date = datetime(2026, 9, 1, 8, 0, 0)
pesanan_list = []
detail_list = []
ds_flat_rows = []
detail_id = 1

for p_id in range(1, 501):
    pelanggan = random.choice(pelanggan_list)
    id_pelanggan = pelanggan[0]

    # Waktu pemesanan acak dalam 80 hari
    tgl_pesan = start_date + timedelta(
        days=random.randint(0, 79),
        hours=random.randint(0, 12),
        minutes=random.randint(0, 59)
    )
    # Waktu dibutuhkan: H+0 (mendadak/urgent) sampai H+7
    selisih_hari = random.choices([0, 1, 2, 3, 5, 7], weights=[15, 30, 25, 15, 10, 5])[0]
    tgl_butuh = tgl_pesan + timedelta(days=selisih_hari, hours=random.randint(1, 5))

    kode = f"NKS-{tgl_pesan.strftime('%Y%m%d')}-{p_id:03d}"

    # Status pesanan (mayoritas selesai karena data historis, sebagian aktif untuk demo)
    if p_id <= 460:
        status = random.choices(["selesai", "dibatalkan"], weights=[94, 6])[0]
    else:
        status = random.choice(["menunggu_konfirmasi", "dikonfirmasi", "diproses", "siap"])

    # Pilih 1 sampai 3 jenis menu dalam 1 pesanan
    item_menus = random.sample(menus, k=random.randint(1, 3))
    total_harga = 0
    total_porsi = 0

    for m in item_menus:
        # Jumlah porsi realistis untuk katering (10 - 150 box)
        jumlah = random.choice([10, 15, 20, 25, 30, 40, 50, 75, 100, 150])
        subtotal = m["harga"] * jumlah
        total_harga += subtotal
        total_porsi += jumlah

        detail_list.append((detail_id, p_id, m["id"], m["harga"], jumlah, subtotal))

        # Baris datar untuk tim DS / AI / Looker Studio
        # Label prioritas (target klasifikasi model AI): Tinggi jika porsi >= 75 atau deadline <= 1 hari
        prioritas = "Tinggi" if (total_porsi >= 75 or selisih_hari <= 1) else ("Sedang" if total_porsi >= 30 else "Normal")

        ds_flat_rows.append({
            "id_pesanan": p_id,
            "kode_pesanan": kode,
            "nama_pelanggan": pelanggan[1],
            "tgl_pemesanan": tgl_pesan.strftime("%Y-%m-%d %H:%M:%S"),
            "tgl_dibutuhkan": tgl_butuh.strftime("%Y-%m-%d %H:%M:%S"),
            "lead_time_hari": selisih_hari,
            "nama_menu": m["nama"],
            "kategori": m["kategori"],
            "harga_satuan": m["harga"],
            "jumlah": jumlah,
            "subtotal": subtotal,
            "status_pesanan": status,
            "label_prioritas_ai": prioritas
        })
        detail_id += 1

    pesanan_list.append((
        p_id, id_pelanggan, kode,
        tgl_pesan.strftime("%Y-%m-%d %H:%M:%S"),
        tgl_butuh.strftime("%Y-%m-%d %H:%M:%S"),
        status, total_harga
    ))

# 4. Export ke file SQL (siap import ke MySQL XAMPP)
with open("seed_transaksi.sql", "w", encoding="utf-8") as f:
    f.write("SET FOREIGN_KEY_CHECKS = 0;\n")
    f.write("TRUNCATE TABLE detail_pesanan;\nTRUNCATE TABLE pesanan;\nTRUNCATE TABLE pelanggan;\n")
    f.write("SET FOREIGN_KEY_CHECKS = 1;\n\n")

    # Insert Pelanggan
    f.write("INSERT INTO pelanggan (id_pelanggan, nama_pelanggan, nomor_whatsapp, alamat) VALUES\n")
    f.write(",\n".join([f"({p[0]}, '{p[1]}', '{p[2]}', '{p[3]}')" for p in pelanggan_list]) + ";\n\n")

    # Insert Pesanan
    f.write("INSERT INTO pesanan (id_pesanan, id_pelanggan, kode_pesanan, tgl_pemesanan, tgl_dibutuhkan, status_pesanan, total_harga) VALUES\n")
    f.write(",\n".join([f"({p[0]}, {p[1]}, '{p[2]}', '{p[3]}', '{p[4]}', '{p[5]}', {p[6]:.2f})" for p in pesanan_list]) + ";\n\n")

    # Insert Detail Pesanan
    f.write("INSERT INTO detail_pesanan (id_detail, id_pesanan, id_menu, harga_satuan, jumlah, subtotal) VALUES\n")
    f.write(",\n".join([f"({d[0]}, {d[1]}, {d[2]}, {d[3]:.2f}, {d[4]}, {d[5]:.2f})" for d in detail_list]) + ";\n")

# 5. Export ke CSV (siap untuk Google Sheets, Looker Studio, & Tim DS/AI)
with open("dataset_pesanan_ds.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=ds_flat_rows[0].keys())
    writer.writeheader()
    writer.writerows(ds_flat_rows)

print("Selesai! File 'seed_transaksi.sql' dan 'dataset_pesanan_ds.csv' berhasil dibuat.")

Selesai! File 'seed_transaksi.sql' dan 'dataset_pesanan_ds.csv' berhasil dibuat.
